# Занятие 5. Кастомная трансформация и объединение данных

> **Цель занятия:** научиться менять структуру таблицы (добавлять/удалять столбцы и строки),
> применять свои функции к данным через `.apply()` и `lambda`, а также **склеивать
> несколько таблиц** в одну через `concat` и `merge`.

**Что будет:**
1. **Работа со столбцами и строками** — добавление, удаление, переименование;
2. **`.apply()` и `lambda`** — как применить свою функцию ко всей колонке или строке;
3. **`.str` accessor** — быстрая работа с текстовыми столбцами;
4. **`pd.concat()`** — склеивание таблиц «стопкой» или «в ширину»;
5. **`pd.merge()`** — соединение таблиц по общему ключу и типы соединений
   (`inner`, `left`, `right`, `outer`).


In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## Загружаем данные про ЕГЭ

Тот же датасет — 1000 учеников, 16 столбцов.

In [ ]:
df = pd.read_csv('data/ege_students.csv', sep=';')
print('Размер таблицы:', df.shape)
df.head()

---
## Часть 1. Работа со столбцами и строками

Прежде чем изменять данные, полезно понимать, **как** менять структуру таблицы:
добавлять новые столбцы, удалять ненужные, переименовывать. Это делают почти в каждом
анализе — сразу после загрузки данных.

### 1.1. Добавляем новый столбец

Самый простой способ — просто **присвоить** значение новому столбцу:

In [ ]:
df_work = df.copy()   # работаем с копией, чтобы не портить оригинал

# новый столбец: во сколько ученик готовится в день
df_work['часов_в_день'] = df_work['часов_подготовки_в_неделю'] / 7
df_work[['имя', 'часов_подготовки_в_неделю', 'часов_в_день']].head()

**Что произошло:**
- слева от `=` мы написали `df_work['часов_в_день']` — Pandas понял: «нужно создать новый столбец с таким именем»;
- справа — формула из уже существующего столбца.

Можно и просто присвоить **одно значение всем строкам** — Pandas сам «растянет» его:

In [ ]:
# добавим столбец-константу
df_work['источник_данных'] = 'ЕГЭ_2026'
df_work.head()

### 1.2. Удаляем столбцы — `.drop()`

Метод **`.drop()`** удаляет колонки. Важно указать `axis=1` — это значит «столбец»
(а `axis=0` было бы «строка»).

In [ ]:
# удалим временный столбец
df_work = df_work.drop('источник_данных', axis=1)
df_work.head()

Можно удалить **несколько столбцов сразу**, передав список:

In [ ]:
# пример: удалим сразу два столбца
tmp = df_work.drop(['часов_в_день', 'ученик_id'], axis=1)
tmp.head()

> **Совет:** `.drop()` **не меняет** исходную таблицу, а возвращает **новую**.
> Поэтому обычно пишут `df = df.drop(...)`, чтобы сохранить результат.
> Если очень хочется поменять «на месте» — есть параметр `inplace=True`, но лучше
> привыкать к явной перезаписи — так меньше сюрпризов.

### 1.3. Переименовываем столбцы — `.rename()`

Передаём **словарь**: `{старое_имя: новое_имя}`.

In [ ]:
# переименуем несколько столбцов
df_work = df_work.rename(columns={
    'часов_подготовки_в_неделю': 'часов_неделя',
    'кол_во_пробников': 'пробников',
})
print('Столбцы теперь:', list(df_work.columns))

### 1.4. Добавляем и удаляем строки

Строки добавляют реже — обычно данные приходят пачкой, и их **склеивают** через
`pd.concat()` (это дальше, в части 4). Здесь только базовые приёмы.

**Удалить строку** по её индексу:

In [ ]:
# удалим строку с меткой 0 (первого ученика)
df_no_first = df_work.drop(0, axis=0)   # 0 - номер строки, axis=0 — строка
print('Было строк:', len(df_work))
print('Стало:', len(df_no_first))
df_no_first.head()

**Удалить несколько строк** — передаём список меток:

In [ ]:
df_short = df_work.drop([0, 1, 2, 3, 4], axis=0)   # удалим первые 5
df_short

> Есть удобная конвенция: если хочешь удалить **пропуски** — используешь `.dropna()`
> (мы разбирали её в прошлом занятии), если удалить **дубликаты** — `.drop_duplicates()`.
> Обычный `.drop()` — только когда точно знаешь, какие метки хочешь убрать.


---
## Часть 2. Свои функции и `.apply()`

Часто нужно применить к столбцу **не готовую операцию** (`+`, `-`, `.mean()`),
а **свою собственную логику**. Например: «поставить категорию по баллу», «взять первую
букву имени», «пересчитать возраст в месяцы». Для этого есть **`.apply()`** — метод,
который **применяет функцию к каждому элементу** столбца (или к каждой строке таблицы).

### 2.1. Простой пример: обычная функция

Напишем функцию, которая по баллу ЕГЭ выдаёт словесную категорию:

In [ ]:
def category(x):
    if x >= 80:
        return 'высокий'
    elif x >= 60:
        return 'средний'
    else:
        return 'низкий'

# применим её к каждому значению столбца
df_work['категория'] = df_work['балл_егэ'].apply(category)
df_work[['имя', 'балл_егэ', 'категория']].head(10)

**Вопрос:** мы вызвали `df_work['балл_егэ'].apply(category)`. Сколько раз Pandas вызвал функцию `category` и что он подставлял в её аргумент `x` на каждом вызове?

### 2.2. Анонимные функции — `lambda`

Если функция очень короткая — писать её отдельно через `def` не хочется. Тогда используют
**`lambda`** — «анонимную функцию», записанную одной строкой:

```python
lambda x: <что вернуть>
```

Это то же самое, что:
```python
def безымянная(x):
    return <что вернуть>
```

**Пример:** сделаем столбец «часов в день» через `lambda`:


In [ ]:
# то же деление на 7, но через lambda прямо в apply
df_work['часов_в_день_v2'] = df_work['часов_неделя'].apply(lambda x: round(x / 7, 2))
df_work[['имя', 'часов_неделя', 'часов_в_день_v2']].head()

**Когда какой способ?**

| Функция | Когда удобно |
|---|---|
| Обычная `def имя(...)` | Логика **сложная**, много строк, нужны условия / несколько операций |
| **`lambda x: ...`** | Логика **простая**, помещается в одну строку |

`lambda` часто пишут прямо внутри `.apply()` — коротко и на месте.

### 2.3. `.apply()` по строкам целиком

Иногда нужно посчитать что-то **из нескольких столбцов сразу** — тогда применяют
функцию к **строке** таблицы, а не к отдельной ячейке. Для этого добавляют `axis=1`:

In [ ]:
# «интегральный балл»: средний балл года × 20 + балл ЕГЭ / 2
df_work['итог'] = df_work.apply(
    lambda row: row['средний_балл_года'] * 20 + row['балл_егэ'] / 2,
    axis=1
).round(1)

df_work[['имя', 'средний_балл_года', 'балл_егэ', 'итог']].head()

**Разница по `axis`:**
- `axis=0` (по умолчанию для `.apply()` на `DataFrame`) — функция получает **столбец целиком**;
- `axis=1` — функция получает **строку целиком** (в виде `Series`), и внутри можно
  обращаться к её значениям как к словарю: `row['имя_столбца']`.


---
## Часть 3. Быстрая работа со строковыми столбцами — `.str`

У каждого текстового столбца в Pandas есть особый «аксессор» **`.str`**, через который
можно применять строковые операции **сразу ко всей колонке** — без `.apply()` и без циклов.

### Примеры:

In [ ]:
# все имена в верхний регистр
df_work['имя'].str.upper().head()

In [ ]:
# длина имени
# df_work['имя']. ...

In [ ]:
# первая буква имени
# df_work['имя']. ...

In [ ]:
# проверяем, содержит ли город слово «Санкт»
df_work['город'].str.contains('Санкт').head()

In [ ]:
# заменим дефис в названии города на пробел
# df_work['город']. ...

**Полезные методы `.str`:**

| Метод | Что делает |
|---|---|
| `.str.upper()` / `.str.lower()` | верхний / нижний регистр |
| `.str.len()` | длина строки |
| `.str.strip()` | убрать пробелы по краям |
| `.str.contains('...')` | содержит ли подстроку (возвращает маску!) |
| `.str.startswith('...')` | начинается ли с подстроки |
| `.str.replace('a', 'b')` | заменить подстроку |
| `.str.split('...')` | разбить по разделителю |
| `.str[0]` | первый символ (как индексация строки) |

`.str.contains(...)` часто комбинируют с булевой маской — это удобный способ фильтрации
по текстовым столбцам:

In [ ]:
# только ученики из городов, в названии которых есть «Санкт» или «Ново»
mask = df_work['город'].str.contains('Санкт|Ново')
df_work[mask]

---
## Часть 4. `pd.concat()` — склеивание таблиц

Часто данные приходят **несколькими кусками**: файл за январь, за февраль, за март.
Их нужно соединить в одну большую таблицу. Для этого используют **`pd.concat()`**.

### 4.1. Склейка «стопкой» (по строкам)

Если таблицы имеют **одинаковые столбцы**, `concat` ставит их друг под другом:

In [ ]:
# разобьём наш датасет на две половины — как будто это два файла
first = df.iloc[:500].copy()
second = df.iloc[500:].copy()

In [ ]:
first

In [ ]:
second

In [ ]:
# склеиваем обратно
combined = pd.concat([first, second])
print('После concat:', combined.shape)
combined.head()

**Вопрос:** посмотрите на индексы в получившейся таблице `combined`. Что с ними не так и почему это может быть проблемой при дальнейшей работе?


In [ ]:
combined = pd.concat([first, second], ignore_index=True)
combined

### 4.2. Склейка «в ширину» (по столбцам)

Если хочется поставить таблицы **рядом**, а не одна под другой, — параметр `axis=1`.
Строки при этом должны быть одинаковыми (тот же индекс).

In [ ]:
# возьмём два кусочка от одной таблицы — как будто у нас два источника про одних людей
left_part  = df[['имя', 'пол', 'возраст']].head()
right_part = df[['балл_егэ', 'сдал']].head()

In [ ]:
left_part

In [ ]:
right_part

In [ ]:
pd.concat([left_part, right_part], axis=1)

**Вопрос:** у вас две таблицы про одних и тех же учеников, но набор столбцов у них разный, и порядок строк тоже отличается. Подойдёт ли здесь `concat` и почему?


---
## Часть 5. `pd.merge()` — соединение по ключу

Самый мощный инструмент — **`pd.merge()`**. Он соединяет две таблицы **по общему столбцу-ключу**,
как `JOIN` в SQL. Классический пример: в основной таблице есть колонка `город`, а в другой
таблице для каждого города записан **регион и население**. Хочется добавить эти сведения
к каждому ученику.

### 5.1. Готовим справочник городов

Сделаем маленькую справочную таблицу:

In [ ]:
cities_info = pd.DataFrame({
    'город': ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск',
              'Екатеринбург', 'Нижний Новгород', 'Самара', 'Ростов-на-Дону'],
    'регион':          ['ЦФО', 'СЗФО', 'ПФО', 'СФО', 'УФО', 'ПФО', 'ПФО', 'ЮФО'],
    'население_млн':   [13.1, 5.6, 1.3, 1.6, 1.5, 1.2, 1.1, 1.1],
})
cities_info

### 5.2. Соединяем таблицы через `merge`

In [ ]:
df_merged = df.merge(cities_info, on='город', how='left')
df_merged[['имя', 'город', 'регион', 'население_млн']].head()

**Вопрос:** мы вызвали `df.merge(cities_info, on='город', how='left')`. Как Pandas решил, какое значение `регион` и `население_млн` поставить в каждую конкретную строку таблицы `df`?


### 5.3. Типы соединений: `how=`

Ключевой параметр `merge` — это **`how`**. Он говорит, **какие строки сохранять**,
если в одной таблице ключ есть, а в другой — нет.

| Тип | Что оставит |
|---|---|
| **`inner`** (по умолчанию) | Только те строки, где **ключ есть в ОБЕИХ** таблицах |
| **`left`** | **Все строки левой** таблицы (правые данные подставляются, если нашлись) |
| **`right`** | **Все строки правой** таблицы (симметрично `left`) |
| **`outer`** | **Все строки из обеих** таблиц (пропуски заполнятся `NaN`) |

Наглядно все четыре варианта через **круги Эйлера**:

![Типы соединений SQL/merge](images/joins_euler.jpg)

Слева всегда — левая таблица, справа — правая. Закрашенная область — те строки,
которые попадут в результат.

**Иллюстрация на маленьком примере.** Сделаем две таблички: в левой — имена (id 1–4),
в правой — города (id 3–6). Общие id — **3 и 4**.

In [ ]:
left = pd.DataFrame({
    'id':   [1, 2, 3, 4],
    'имя':  ['Аня', 'Боря', 'Вика', 'Гена'],
})
right = pd.DataFrame({
    'id':    [3, 4, 5, 6],
    'город': ['Москва', 'Казань', 'Самара', 'Уфа'],
})
print('LEFT:'); print(left)
print('\nRIGHT:'); print(right)

#### `inner` — только общие ключи

Остаются только те строки, у которых `id` **есть в обеих таблицах** — то есть 3 и 4.
Остальные строки (id 1, 2, 5, 6) отбрасываются.

![inner join](images/merge_inner.png)

In [ ]:
# inner — только id, которые есть В ОБЕИХ таблицах (3 и 4)
left.merge(right, on='id', how='inner')

#### `left` — все строки левой таблицы

Берём **все имена** (id 1–4). Где в правой таблице нашёлся город — подставляем его (id 3, 4),
где не нашёлся (id 1, 2) — пишем `NaN`. Строки из правой таблицы, которых нет слева (id 5, 6),
теряются.

![left join](images/merge_left.png)

In [ ]:
# left — все имена, город подставится только для id 3 и 4
left.merge(right, on='id', how='left')

#### `right` — все строки правой таблицы

Зеркально `left`: берём **все города** (id 3–6), а имя подставляется только для 3 и 4 — остальное `NaN`.
Строки из левой, которых нет справа (id 1, 2), теряются.

![right join](images/merge_right.png)

In [ ]:
# right — все города, имя подставится только для id 3 и 4
left.merge(right, on='id', how='right')

#### `outer` — все ключи из обеих таблиц

Не теряем ничего: в результате будут все id (1–6). Где в одной из таблиц данных нет — ставим `NaN`.

![outer join](images/merge_outer.png)

In [ ]:
# outer — вообще все id из обеих таблиц
left.merge(right, on='id', how='outer')

### 5.4. Что если ключи называются по-разному?

До сих пор в обеих таблицах ключевой столбец назывался одинаково — `город`. Тогда
хватало `on='город'`. Но часто на практике **имена не совпадают**: в основной таблице —
`город`, а в справочнике — `city`. Переименовывать необязательно — можно указать
**два ключа** через `left_on=` и `right_on=`.


In [ ]:
students = pd.DataFrame({
    'имя':   ['Аня', 'Боря', 'Вика', 'Гена'],
    'город': ['Москва', 'Казань', 'Уфа', 'Москва'],
})

cities_info = pd.DataFrame({
    'city':   ['Москва', 'Казань', 'Уфа', 'Пермь'],
    'регион': ['Центр',  'Поволжье', 'Урал', 'Урал'],
})

students.merge(cities_info, left_on='город', right_on='city', how='left')


**Что произошло:** Pandas взял значение из столбца `город` слева, нашёл такое же в столбце
`city` справа и подтянул `регион`. Оба столбца-ключа при этом **остаются** в результате
(и `город`, и `city`) — один из них можно сразу удалить через `.drop(columns='city')`,
если он не нужен.

> Если ключевые столбцы называются **одинаково** — короче писать просто `on='...'`.


### 5.5. Соединение сразу по нескольким столбцам

Иногда одного ключа **не хватает**, чтобы однозначно определить строку. Классический
пример: у одного ученика есть оценки по разным предметам, и учитель для «Ани по математике»
и «Ани по русскому» — **разный**. Тогда ключ — это **пара** `(имя, предмет)`.

В `merge` можно передать **список столбцов** — Pandas будет искать совпадение сразу по всем.


In [ ]:
scores = pd.DataFrame({
    'имя':     ['Аня', 'Аня', 'Боря', 'Боря', 'Вика'],
    'предмет': ['матем', 'рус', 'матем', 'рус', 'матем'],
    'балл':    [82, 74, 60, 55, 91],
})

teachers = pd.DataFrame({
    'имя':     ['Аня', 'Аня', 'Боря', 'Боря'],
    'предмет': ['матем', 'рус', 'матем', 'рус'],
    'учитель': ['Иванов', 'Петров', 'Иванов', 'Сидоров'],
})

scores.merge(teachers, on=['имя', 'предмет'], how='left')


**Что произошло:** Pandas для каждой строки `scores` искал в `teachers` строку с **точно
такой же парой** `(имя, предмет)`. Для Ани по математике нашёл Иванова, для Ани по русскому —
Петрова, хотя имя то же самое. Для Вики по математике совпадения не нашлось — в результате `NaN`.

> Если бы мы соединили только по `имя`, для «Ани по математике» Pandas нашёл бы **обе** её
> строки в `teachers` — и результат бы «раздулся» с дубликатами. Правильно выбранный набор
> ключей — это то, что делает `merge` **однозначным**.


### 5.6. Когда что использовать?

**Вопрос:** у вас есть таблица `students` (основная) и справочник `regions` — регион для каждого города. Вы хотите добавить регион к каждому ученику, ничего не потеряв из `students`. Какой `how=` выберете — и почему не `inner`?


---
## Итог занятия

Сегодня научились:
- **менять структуру таблицы** — добавлять столбцы через `df['новый'] = ...`,
  удалять через `.drop(..., axis=1)`, переименовывать через `.rename(columns={...})`;
- применять **свои функции** через **`.apply()`** — по столбцу или по строке (`axis=1`);
- использовать **`lambda`** для коротких функций прямо на месте;
- быстро обрабатывать текстовые столбцы через **`.str.upper()`, `.str.contains(...)`** и другие;
- **склеивать таблицы** через `pd.concat()` (когда столбцы одинаковые);
- **соединять таблицы по ключу** через `pd.merge()`: 4 типа соединения (`inner`, `left`, `right`, `outer`), а также `left_on=` / `right_on=` для ключей с разными именами и `on=[col1, col2]` для соединения сразу по нескольким столбцам.

**Главная мысль:** реальный анализ почти всегда — это **не одна таблица**, а **несколько**,
которые нужно **преобразовать**, **склеить** и **обогатить** справочными данными.
